# S4 · AndinaLog 03B · Notebook 2 · Tratamiento de Productos

Este notebook toma las salidas del diagnóstico v2 de `andinalog_productos.csv` y un catálogo de reglas. Conserva las 63 filas originales en el archivo tratado y genera un Silver solo con productos utilizables. No modifica el Bronze. Cada tratamiento se aplica únicamente cuando su regla figura `APROBADA` y tiene evidencia documentada.

El catálogo inicial tiene todas las reglas en `PENDIENTE`. Con ese estado, el notebook puede ejecutarse y produce un Silver de las filas sin problemas, sin corregir las demás. Antes de aprobar una regla, confirma su criterio y registra la evidencia en el CSV.


## 1 · Rutas y entradas

En local, ejecuta desde `practicasNotebookColab`. En Colab, ajusta la carpeta de Drive que contiene `datasets/` y `proyecto-integrador/`. Coloca `catalogo_reglas_tratamiento_productos.csv` en `proyecto-integrador/andinalog_productos/notebook2/` antes de ejecutar.


In [2]:
from pathlib import Path
import hashlib
import os
import tempfile
import pandas as pd

ENTORNO = "auto"  # auto, local o drive
RUTA_PROYECTO_DRIVE = "/content/drive/MyDrive/GIAD"
VERSION_TRATAMIENTO = "GIAD-M3-S4-productos-tratamiento-v1"
VERSION_DIAGNOSTICO_REQUERIDA = "GIAD-M3-S4-productos-diagnostico-v2"
COLUMNAS_BRONZE = ["producto_id", "nombre_producto", "categoria_logistica",
                   "temperatura_conservacion_requerida_c", "tolerancia_temperatura_c",
                   "precio_unitario_bob", "costo_unitario_bob"]
NUMERICAS = ["temperatura_conservacion_requerida_c", "tolerancia_temperatura_c",
            "precio_unitario_bob", "costo_unitario_bob"]

def raiz_local():
    for carpeta in [Path.cwd(), *Path.cwd().parents]:
        if ((carpeta / "datasets" / "AndinaLog_03B_Bronce").is_dir()
                and (carpeta / "proyecto-integrador").is_dir()):
            return carpeta
    raise FileNotFoundError("Ejecuta dentro de practicasNotebookColab")

def configurar_rutas(entorno, ruta_drive):
    if entorno == "auto":
        entorno = "drive" if "google.colab" in __import__("sys").modules else "local"
    if entorno == "drive":
        from google.colab import drive
        drive.mount("/content/drive")
        raiz = Path(ruta_drive)
    elif entorno == "local":
        raiz = raiz_local()
    else:
        raise ValueError("ENTORNO debe ser auto, local o drive")
    caso = raiz / "proyecto-integrador" / "andinalog_productos"
    return {
        "bronze": raiz / "datasets" / "AndinaLog_03B_Bronce" / "andinalog_productos.csv",
        "principal": caso / "notebook1" / "salidas" / "andinalog_productos_diagnosticado.csv",
        "problemas": caso / "notebook1" / "salidas" / "andinalog_productos_problemas.csv",
        "reporte1": caso / "notebook1" / "salidas" / "andinalog_productos_reporte_calidad.csv",
        "catalogo": caso / "notebook2" / "catalogo_reglas_tratamiento_productos.csv",
        "salidas": caso / "notebook2" / "salidas",
    }

rutas = configurar_rutas(ENTORNO, RUTA_PROYECTO_DRIVE)
for nombre in ["bronze", "principal", "problemas", "reporte1", "catalogo"]:
    if not rutas[nombre].is_file():
        raise FileNotFoundError(f"Falta {nombre}: {rutas[nombre]}")
print("Bronze:", rutas["bronze"])
print("Salidas:", rutas["salidas"])


Bronze: c:\Users\remrodri\Github\practicasNotebookColab\datasets\AndinaLog_03B_Bronce\andinalog_productos.csv
Salidas: c:\Users\remrodri\Github\practicasNotebookColab\proyecto-integrador\andinalog_productos\notebook2\salidas


## 2 · Lectura y validación

El SHA-256 y la versión del reporte deben corresponder al Bronze y al diagnóstico actuales. Si falla la comprobación, vuelve a ejecutar el notebook 1 antes de tratar Productos.


In [3]:
def leer_csv(ruta):
    return pd.read_csv(ruta, dtype="string", encoding="utf-8-sig", keep_default_na=False)

principal = leer_csv(rutas["principal"])
problemas = leer_csv(rutas["problemas"])
reporte1 = leer_csv(rutas["reporte1"])
catalogo = leer_csv(rutas["catalogo"])
bronze = leer_csv(rutas["bronze"])

if list(bronze.columns) != COLUMNAS_BRONZE:
    raise ValueError("Esquema Bronze inesperado")
if not {"fila_bronze", *COLUMNAS_BRONZE, "en_cuarentena"}.issubset(principal.columns):
    raise ValueError("El diagnosticado carece de columnas obligatorias")
if not {"fila_bronze", "columna_afectada", "codigo_error", "valor_original", "version_diagnostico"}.issubset(problemas.columns):
    raise ValueError("El detalle de problemas no cumple el contrato")
if not {"regla_id", "estado", "evidencia_acuerdo"}.issubset(catalogo.columns):
    raise ValueError("El catálogo no cumple el contrato")
if principal["fila_bronze"].duplicated().any() or catalogo["regla_id"].duplicated().any():
    raise ValueError("Identificadores internos duplicados")
if not catalogo["estado"].isin(["APROBADA", "PENDIENTE"]).all():
    raise ValueError("Estado de regla desconocido")
if not problemas["fila_bronze"].isin(principal["fila_bronze"]).all():
    raise ValueError("Problemas sin fila diagnosticada")
resumen = reporte1.set_index("metrica")["valor"]
HASH_BRONZE = hashlib.sha256(rutas["bronze"].read_bytes()).hexdigest()
if resumen.loc["sha256_bronze"] != HASH_BRONZE:
    raise ValueError("Bronze distinto del diagnóstico; ejecuta notebook 1")
if resumen.loc["version_diagnostico"] != VERSION_DIAGNOSTICO_REQUERIDA:
    raise ValueError("Se requiere el diagnóstico v2 de Productos")
if len(bronze) != len(principal) or len(bronze) != int(resumen.loc["filas_bronze"]):
    raise ValueError("Conteo de filas incompatible")
if len(problemas) != int(resumen.loc["problemas_detectados"]):
    raise ValueError("Conteo de problemas incompatible")
pd.testing.assert_frame_equal(principal[COLUMNAS_BRONZE].reset_index(drop=True), bronze.reset_index(drop=True))
if set(problemas["fila_bronze"]) != set(principal.loc[principal["en_cuarentena"].str.lower().eq("true"), "fila_bronze"]):
    raise ValueError("Cuarentena y problemas no coinciden")
print(f"Entradas válidas: {len(bronze)} filas; {len(problemas)} problemas; SHA-256 {HASH_BRONZE}")
display(catalogo)


Entradas válidas: 63 filas; 14 problemas; SHA-256 0ee6adc9905e0d41ec5a2646f64633cccbab53ac35ee66a95f9bd46687744eda


,regla_id,columna_afectada,codigo_error,tratamiento_propuesto,estado,evidencia_acuerdo,validacion_requerida
0,ID_NORMALIZAR,producto_id,FORMATO_INVALIDO,Quitar espacios exteriores y pasar a mayúscula...,PENDIENTE,,Aprobar equivalencia y comprobar unicidad tras...
1,CATEGORIA_CONJELADO,categoria_logistica,VALOR_NO_RECONOCIDO,Corregir el literal conjelado a Congelado,PENDIENTE,,Confirmar equivalencia con la fuente o respons...
2,DUPLICADO_IDENTICO,producto_id,DUPLICADO,Conservar primera fila y excluir copias con lo...,PENDIENTE,,Aprobar criterio de canonicidad y verificar ig...
3,COLISION_ID_NORMALIZADO,producto_id,COLISION_ID_NORMALIZADO,"Tras normalizar el ID, excluir la copia solo s...",PENDIENTE,,Requiere ID_NORMALIZAR aprobada y comparación ...
4,TEMP_FALTANTE,temperatura_conservacion_requerida_c,FALTANTE,Mantener en cuarentena hasta recuperar valor d...,PENDIENTE,,Valor original verificable; no imputar por cat...


## 3 · Reglas aprobadas y decisiones sobre IDs repetidos

`PROD-010` y `PROD-014` tienen copias idénticas; ` prod-024 ` colisiona con `PROD-024` después de normalizar. La selección de una fila canónica solo ocurre cuando las reglas correspondientes están aprobadas. Si un grupo tiene datos contradictorios, queda pendiente.


In [4]:
def aprobada(regla_id):
    fila = catalogo.loc[catalogo["regla_id"].eq(regla_id)]
    if len(fila) != 1:
        raise ValueError(f"Falta regla única: {regla_id}")
    ok = fila.iloc[0]["estado"] == "APROBADA"
    if ok and not str(fila.iloc[0]["evidencia_acuerdo"]).strip():
        raise ValueError(f"Regla {regla_id} aprobada sin evidencia")
    return ok

AUTORIZADAS = {rid: aprobada(rid) for rid in ["ID_NORMALIZAR", "CATEGORIA_CONJELADO",
                                             "DUPLICADO_IDENTICO", "COLISION_ID_NORMALIZADO"]}

df_trabajo = principal.copy(deep=True)
id_original = df_trabajo["producto_id"]
id_candidato = id_original.str.strip().str.upper()
formato_valido = id_candidato.str.fullmatch(r"PROD-\d{3}").fillna(False)
df_trabajo["producto_id_preparado"] = id_original.copy()
df_trabajo["categoria_logistica_preparada"] = df_trabajo["categoria_logistica"].copy()
if AUTORIZADAS["ID_NORMALIZAR"]:
    cambiar = formato_valido & id_original.ne(id_candidato)
    df_trabajo.loc[cambiar, "producto_id_preparado"] = id_candidato[cambiar]
if AUTORIZADAS["CATEGORIA_CONJELADO"]:
    cambiar = df_trabajo["categoria_logistica"].eq("conjelado")
    df_trabajo.loc[cambiar, "categoria_logistica_preparada"] = "Congelado"

def decidir_duplicados(df):
    copia = df.copy()
    copia["_id_candidato"] = copia["producto_id"].str.strip().str.upper()
    decisiones = []
    for clave, grupo in copia.groupby("_id_candidato", sort=False):
        if len(grupo) < 2:
            continue
        grupo = grupo.sort_values("fila_bronze", key=lambda s: s.astype(int))
        todos_iguales = grupo[COLUMNAS_BRONZE].nunique(dropna=False).eq(1).all()
        otros_iguales = grupo[[c for c in COLUMNAS_BRONZE if c != "producto_id"]].nunique(dropna=False).eq(1).all()
        if todos_iguales:
            tipo = "IDENTICO"
            autorizado = AUTORIZADAS["DUPLICADO_IDENTICO"]
        elif otros_iguales:
            tipo = "ID_EQUIVALENTE"
            autorizado = AUTORIZADAS["ID_NORMALIZAR"] and AUTORIZADAS["COLISION_ID_NORMALIZADO"]
        else:
            tipo = "CONFLICTO"
            autorizado = False
        canonica = grupo.iloc[0]["fila_bronze"] if autorizado else ""
        for _, fila in grupo.iterrows():
            decision = ("CANONICA" if fila["fila_bronze"] == canonica else "COPIA_EXCLUIDA") if autorizado else "PENDIENTE"
            decisiones.append({"fila_bronze": fila["fila_bronze"], "producto_id_normalizado": clave,
                               "tipo_duplicado": tipo, "decision_duplicado": decision,
                               "fila_canonica": canonica,
                               "justificacion": "Campos coincidentes y regla aprobada" if autorizado else "Sin regla aprobada o datos en conflicto"})
    return pd.DataFrame(decisiones, columns=["fila_bronze", "producto_id_normalizado", "tipo_duplicado",
                                             "decision_duplicado", "fila_canonica", "justificacion"])

decisiones = decidir_duplicados(df_trabajo)
display(decisiones)


,fila_bronze,producto_id_normalizado,tipo_duplicado,decision_duplicado,fila_canonica,justificacion
0,10,PROD-010,IDENTICO,PENDIENTE,,Sin regla aprobada o datos en conflicto
1,61,PROD-010,IDENTICO,PENDIENTE,,Sin regla aprobada o datos en conflicto
2,14,PROD-014,IDENTICO,PENDIENTE,,Sin regla aprobada o datos en conflicto
3,63,PROD-014,IDENTICO,PENDIENTE,,Sin regla aprobada o datos en conflicto
4,24,PROD-024,ID_EQUIVALENTE,PENDIENTE,,Sin regla aprobada o datos en conflicto
5,62,PROD-024,ID_EQUIVALENTE,PENDIENTE,,Sin regla aprobada o datos en conflicto


## 4 · Acciones, cuarentena final y Silver

Una fila sale de cuarentena solo si todos sus problemas se resuelven. Las copias excluidas quedan en el archivo tratado y en cuarentena para auditoría; el Silver nunca cuenta una copia como producto adicional.


In [5]:
acciones = problemas.copy(deep=True)
acciones["estado_tratamiento"] = "PENDIENTE"
acciones["tratamiento_aplicado"] = "NINGUNO"
acciones["detalle_resultado"] = "Sin regla aprobada o evidencia suficiente"
por_fila = df_trabajo.set_index("fila_bronze")
if AUTORIZADAS["ID_NORMALIZAR"]:
    id_preparado = acciones["fila_bronze"].map(por_fila["producto_id_preparado"])
    id_original_accion = acciones["fila_bronze"].map(por_fila["producto_id"])
    normalizado = (acciones["codigo_error"].eq("FORMATO_INVALIDO")
                   & id_preparado.str.fullmatch(r"PROD-\d{3}").fillna(False)
                   & id_preparado.ne(id_original_accion))
    acciones.loc[normalizado, ["estado_tratamiento", "tratamiento_aplicado", "detalle_resultado"]] = [
        "RESUELTO", "ID_NORMALIZADO", "Espacios y mayúsculas normalizados según regla aprobada"]
if AUTORIZADAS["CATEGORIA_CONJELADO"]:
    categoria = acciones["fila_bronze"].map(por_fila["categoria_logistica"])
    corregida = acciones["codigo_error"].eq("VALOR_NO_RECONOCIDO") & categoria.eq("conjelado")
    acciones.loc[corregida, ["estado_tratamiento", "tratamiento_aplicado", "detalle_resultado"]] = [
        "RESUELTO", "CATEGORIA_CORREGIDA", "conjelado confirmado como Congelado"]
por_decision = decisiones.set_index("fila_bronze")["decision_duplicado"]
decision = acciones["fila_bronze"].map(por_decision)
problema_dup = acciones["codigo_error"].isin(["DUPLICADO", "COLISION_ID_NORMALIZADO"])
canonica = problema_dup & decision.eq("CANONICA")
copia = problema_dup & decision.eq("COPIA_EXCLUIDA")
acciones.loc[canonica, ["estado_tratamiento", "tratamiento_aplicado", "detalle_resultado"]] = [
    "RESUELTO", "SELECCION_CANONICA", "Fila canónica según regla aprobada"]
acciones.loc[copia, ["estado_tratamiento", "tratamiento_aplicado", "detalle_resultado"]] = [
    "EXCLUIDO_COMO_COPIA", "COPIA_EXCLUIDA", "Copia conservada para auditoría"]
# Asegurar que cualquier copia aprobada quede excluida aunque no tuviera hallazgo previo.
ya_marcadas = set(acciones.loc[copia, "fila_bronze"])
extra = decisiones.loc[decisiones["decision_duplicado"].eq("COPIA_EXCLUIDA")
                      & ~decisiones["fila_bronze"].isin(ya_marcadas)]
if len(extra):
    adicionales = pd.DataFrame({"fila_bronze": extra["fila_bronze"],
        "columna_afectada": "producto_id", "codigo_error": "COPIA_EXCLUIDA",
        "valor_original": extra["producto_id_normalizado"],
        "version_diagnostico": VERSION_DIAGNOSTICO_REQUERIDA,
        "estado_tratamiento": "EXCLUIDO_COMO_COPIA", "tratamiento_aplicado": "COPIA_EXCLUIDA",
        "detalle_resultado": "Copia conservada para auditoría"})
    acciones = pd.concat([acciones, adicionales], ignore_index=True)
acciones = acciones.sort_values(["fila_bronze", "codigo_error"],
                                key=lambda s: s.astype(int) if s.name == "fila_bronze" else s,
                                kind="stable").reset_index(drop=True)

df_final = df_trabajo.copy(deep=True)
for campo in ["tipo_duplicado", "decision_duplicado", "fila_canonica", "justificacion"]:
    df_final[campo] = df_final["fila_bronze"].map(decisiones.set_index("fila_bronze")[campo]).fillna("")
df_final["en_cuarentena_inicial"] = df_final["en_cuarentena"].str.lower().eq("true")
pendientes = acciones.loc[acciones["estado_tratamiento"].ne("RESUELTO")].copy()
pendientes["motivo"] = pendientes["columna_afectada"] + ":" + pendientes["codigo_error"]
motivos = pendientes.groupby("fila_bronze")["motivo"].agg(lambda x: "|".join(dict.fromkeys(x)))
df_final["motivos_finales"] = df_final["fila_bronze"].map(motivos).fillna("")
df_final["en_cuarentena_final"] = df_final["motivos_finales"].ne("")
df_final["version_tratamiento"] = VERSION_TRATAMIENTO
cuarentena_final = df_final.loc[df_final["en_cuarentena_final"]].copy()
silver = df_final.loc[~df_final["en_cuarentena_final"], ["producto_id_preparado", "nombre_producto",
    "categoria_logistica_preparada", *NUMERICAS]].copy()
silver.columns = COLUMNAS_BRONZE
for col in NUMERICAS:
    silver[col] = pd.to_numeric(silver[col].str.strip(), errors="coerce")

pd.testing.assert_frame_equal(df_final[COLUMNAS_BRONZE].reset_index(drop=True), bronze.reset_index(drop=True))
assert len(df_final) == len(bronze)
assert len(cuarentena_final) == int(df_final["en_cuarentena_final"].sum())
assert len(silver) + len(cuarentena_final) == len(bronze)
assert not silver["producto_id"].duplicated().any()
assert silver[NUMERICAS].notna().all().all()
assert silver["producto_id"].str.fullmatch(r"PROD-\d{3}").all()
assert silver["categoria_logistica"].isin(["Fresco", "Congelado", "Seco"]).all()
assert set(pendientes["fila_bronze"]) == set(cuarentena_final["fila_bronze"])
assert df_final.loc[df_final["decision_duplicado"].eq("COPIA_EXCLUIDA"), "en_cuarentena_final"].all()
print("Silver:", len(silver), "productos; cuarentena final:", len(cuarentena_final))
display(acciones.groupby(["codigo_error", "estado_tratamiento"]).size().rename("filas").reset_index())


Silver: 50 productos; cuarentena final: 13


,codigo_error,estado_tratamiento,filas
0,COLISION_ID_NORMALIZADO,PENDIENTE,1
1,DUPLICADO,PENDIENTE,2
2,FALTANTE,PENDIENTE,2
3,FORMATO_INVALIDO,PENDIENTE,6
4,VALOR_NO_RECONOCIDO,PENDIENTE,3


## 5 · Reporte y exportación

Las salidas se generan únicamente después de validar resultados y comprobar de nuevo que el Bronze no cambió. El reporte enumera estados de reglas y cifras reales de esta ejecución.


In [6]:
def crear_reporte():
    datos = [
        ("sha256_bronze", HASH_BRONZE),
        ("version_diagnostico_origen", VERSION_DIAGNOSTICO_REQUERIDA),
        ("version_tratamiento", VERSION_TRATAMIENTO),
        ("filas_bronze", len(bronze)),
        ("filas_cuarentena_inicial", int(df_final["en_cuarentena_inicial"].sum())),
        ("filas_cuarentena_final", len(cuarentena_final)),
        ("filas_liberadas", int((df_final["en_cuarentena_inicial"] & ~df_final["en_cuarentena_final"]).sum())),
        ("filas_silver", len(silver)),
        ("problemas_resueltos", int(acciones["estado_tratamiento"].eq("RESUELTO").sum())),
        ("copias_excluidas", int(acciones["estado_tratamiento"].eq("EXCLUIDO_COMO_COPIA").sum())),
        ("problemas_pendientes", int(acciones["estado_tratamiento"].eq("PENDIENTE").sum())),
    ]
    datos.extend(("regla_" + r["regla_id"], r["estado"]) for _, r in catalogo.iterrows())
    return pd.DataFrame(datos, columns=["metrica", "valor"])

reporte2 = crear_reporte()

def exportar(directorio, tablas):
    if hashlib.sha256(rutas["bronze"].read_bytes()).hexdigest() != HASH_BRONZE:
        raise RuntimeError("El Bronze cambió durante la ejecución; se cancela la exportación")
    directorio.mkdir(parents=True, exist_ok=True)
    temporales = {}
    try:
        for nombre, tabla in tablas.items():
            destino = directorio / nombre
            with tempfile.NamedTemporaryFile(mode="w", suffix=".csv", prefix=".tmp_productos2_",
                                             dir=directorio, encoding="utf-8-sig", newline="", delete=False) as tmp:
                tabla.to_csv(tmp, index=False)
                temporales[destino] = Path(tmp.name)
        for destino, temporal in temporales.items():
            os.replace(temporal, destino)
    finally:
        for temporal in temporales.values():
            temporal.unlink(missing_ok=True)
    return list(temporales)

tablas = {
    "andinalog_productos_tratado.csv": df_final,
    "andinalog_productos_silver.csv": silver,
    "andinalog_productos_acciones.csv": acciones,
    "andinalog_productos_decisiones_duplicados.csv": decisiones,
    "andinalog_productos_cuarentena_final.csv": cuarentena_final,
    "andinalog_productos_reporte_tratamiento.csv": reporte2,
}
for ruta in exportar(rutas["salidas"], tablas):
    print(ruta)
display(reporte2)


c:\Users\remrodri\Github\practicasNotebookColab\proyecto-integrador\andinalog_productos\notebook2\salidas\andinalog_productos_tratado.csv
c:\Users\remrodri\Github\practicasNotebookColab\proyecto-integrador\andinalog_productos\notebook2\salidas\andinalog_productos_silver.csv
c:\Users\remrodri\Github\practicasNotebookColab\proyecto-integrador\andinalog_productos\notebook2\salidas\andinalog_productos_acciones.csv
c:\Users\remrodri\Github\practicasNotebookColab\proyecto-integrador\andinalog_productos\notebook2\salidas\andinalog_productos_decisiones_duplicados.csv
c:\Users\remrodri\Github\practicasNotebookColab\proyecto-integrador\andinalog_productos\notebook2\salidas\andinalog_productos_cuarentena_final.csv
c:\Users\remrodri\Github\practicasNotebookColab\proyecto-integrador\andinalog_productos\notebook2\salidas\andinalog_productos_reporte_tratamiento.csv


,metrica,valor
0,sha256_bronze,0ee6adc9905e0d41ec5a2646f64633cccbab53ac35ee66...
1,version_diagnostico_origen,GIAD-M3-S4-productos-diagnostico-v2
2,version_tratamiento,GIAD-M3-S4-productos-tratamiento-v1
3,filas_bronze,63
4,filas_cuarentena_inicial,13
5,filas_cuarentena_final,13
6,filas_liberadas,0
7,filas_silver,50
8,problemas_resueltos,0
9,copias_excluidas,0


## Siguiente etapa

Revisa el catálogo y el archivo de acciones antes de afirmar que un producto fue tratado. Las reglas pendientes conservan las filas en cuarentena. El Silver producido con todas las reglas pendientes contiene únicamente los productos que ya pasaron el diagnóstico sin problemas.
